In [ ]:
!pip install torch opencv-python imageio mediapipe

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# import torchvision

import cv2
# import tensorflow as tf

import numpy as np
from typing import List
from matplotlib import pyplot as plt
import imageio


In [1]:
from processing.video_loader import VideoLoader
from processing.preprocessor import Preprocessor

# building data loading functions

download dataset from kaggle
https://www.kaggle.com/datasets/mohamedbentalb/lipreading-dataset and save them in data\lips-dataset

In [ ]:
#test_path = r'.\data\lips-dataset\s1\bbaf2n.mpg' ## Windows File Directory
test_path = r'./data/lips-dataset/s1/bbaf2n.mpg' ## Macbook File Directory
video_loader = VideoLoader()
video_data = video_loader.load(test_path)
# video_data

In [ ]:
preprocessor = Preprocessor()
video_data = preprocessor.convert_to_greyscale(video_data)

if fail to crop lips, head over to google https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task



In [ ]:
# cv2.imshow("greyscale", video_data["frames"][0])
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [ ]:
vocab = [x for x in "abcdefghijklmnopqrstuvwxyz'?!123456789 "]

In [ ]:
char_to_num = {char: idx for idx, char in enumerate(vocab)}
num_to_char = {idx: char for char, idx in char_to_num.items()}

print(f"The vocabulary is: {vocab} (size = {len(vocab)})")

In [ ]:
def encode_text(text):
    return torch.tensor([char_to_num[c] for c in text], dtype=torch.long)

def decode_indices(indices):
    return "".join(num_to_char[int(i)] for i in indices)

x = encode_text("hello")
print(x)
print(decode_indices(x))

In [ ]:
def load_alignments(path:str) -> List[str]: 
    with open(path, 'r') as f: 
        lines = f.readlines() 
    tokens = []
    for line in lines:
        line = line.split()
        if line[2] != 'sil': 
            tokens = [*tokens,' ',line[2]]
    return [encode_text(x) for x in tokens]

def load_data(path: str) -> dict: 
    # path = bytes.decode(path.numpy())
    #file_name = path.split('/')[-1].split('.')[0]
    # File name splitting for windows
    file_name = os.path.basename(path).split('.')[0]
    video_path = os.path.join('data','lips-dataset','s1',f'{file_name}.mpg')
    alignment_path = os.path.join('data', 'lips-dataset', 'alignments','s1',f'{file_name}.align')
    video_data = video_loader.load(path) 
    alignments = load_alignments(alignment_path)
    video_data['alignments'] = alignments
    return video_data


video_data = load_data(test_path)
# print(video_data)
print(video_data['alignments'])
print([decode_indices(x) for x in video_data['alignments']])

# Creating data pipeline

In [ ]:
samples  = []
word_dict = {}
for i in os.listdir('data/lips-dataset/s1'):
    if i.endswith('.mpg'):
        print(i)
        video_data = load_data(os.path.join('data/lips-dataset/s1', i))
        samples.append((video_data['frames'], video_data['alignments']))
        
        print(video_data['alignments'])
        sentence = [decode_indices(x) for x in video_data['alignments']]
        print(sentence)
        for word in sentence: 
            if word not in word_dict: 
                word_dict[word] = 1
            else: 
                word_dict[word] += 1
    break

print(sorted(word_dict.items(), key=lambda x: x[1], reverse=True))

In [ ]:
cv2.imshow("test", video_data['frames'][40])
cv2.waitKey(0)
cv2.destroyAllWindows()

## Alignment Algorithm
### LBY: The cell below runs the align_videos.py as if it was in a terminal, input arguments include data directory and output directory
### It is important to note to save the kaggle dataset in its subfolder called data/lips-dataset, of which should have the following subfolders:
### (a) alignments, (b) s1, (c) processedData [This is an addition]

In [ ]:
!python align_videos.py --video-dir data/lips-dataset/s1 --align-dir data/lips-dataset/alignments/s1 --output-dir data/lips-dataset/processedData --grayscale --crop-mouth

## Train/Test Split & Model Evaluation
### LBY: Split `aligned_words.csv` (word-level, stratified 80/20), train a LipNet-style classifier, and report test accuracy.


In [3]:
from dataset import get_word_classification_dataloaders
from model import LipReadingClassifier

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

train_loader, test_loader, word_to_index, index_to_word = get_word_classification_dataloaders(
    test_size=0.2, seed=42, batch_size=32, stratify=True
)
num_classes = len(word_to_index)

print(f'Device: {device}')
print(f'Train samples: {len(train_loader.dataset)}')
print(f'Test samples: {len(test_loader.dataset)}')
print(f'Classes: {num_classes}')


Device: mps
Train samples: 4796
Test samples: 1204
Classes: 51


In [4]:
import torch.nn as nn

model = LipReadingClassifier(num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


def evaluate(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for frames, labels in loader:
            frames, labels = frames.to(device), labels.to(device)
            predictions = model(frames).argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total


epochs = 5  # increase for higher accuracy (30 epochs reached ~81%)
for epoch in range(1, epochs + 1):
    model.train()
    for frames, labels in train_loader:
        frames, labels = frames.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(frames), labels)
        loss.backward()
        optimizer.step()
    test_accuracy = evaluate(test_loader)
    print(f'Epoch {epoch}/{epochs} | test accuracy: {test_accuracy:.4f}')

print(f'Final test accuracy: {evaluate(test_loader):.4f}')


Epoch 1/5 | test accuracy: 0.1304
Epoch 2/5 | test accuracy: 0.1412
Epoch 3/5 | test accuracy: 0.1520
Epoch 4/5 | test accuracy: 0.1653
Epoch 5/5 | test accuracy: 0.1694
Final test accuracy: 0.1694


In [5]:
# Optionally load the checkpoint saved by train.py and re-evaluate.
checkpoint = torch.load('checkpoints/best.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
saved_epoch = checkpoint['epoch']
saved_acc = checkpoint['test_accuracy']
print(f'Loaded checkpoint (epoch {saved_epoch}, saved test acc {saved_acc:.4f})')
print(f'Re-evaluated test accuracy: {evaluate(test_loader):.4f}')


Loaded checkpoint (epoch 29, saved test acc 0.8098)
Re-evaluated test accuracy: 0.8098
